# Encoding

## Quantum Random Access Memory

A classical RAM associates information with memory addresses such that $\ \text{RAM:}\ i \rightarrow D[i]$.
This way, given a set of addresses, we can retrieve one piece of information per access.

Meanwhile, a quantum RAM (QRAM) can associate different addresses with information in superposition. $\ \text{QRAM}:\ \sum_{i=0}^{n-1} \alpha_i |i\rangle |0\rangle \rightarrow \sum_{i=0}^{n-1} \alpha_i |i\rangle |D[i]\rangle $.
In other words, a QRAM allows access to multiple memory locations in superposition within a single operation.

The memory can be addressed with $O(logn)$ qubits, so the total number of qubits required grows only logarithmically with the number of memory locations, allowing us to index large datasets efficiently.

The addressing is compatible with base, phase, angle and amplitude encoding.

There are different ways to implement a QRAM. We can use multicontroled gates in order to load a data item if and only if a binary memory direction is given.
It is necessary to apply Hadamard gates in order to have in superposition all the possible directions.

![alt text](<qram.png>)

## Angle Encoding

In this algorithm, rotation gates are used to encode the data.

For each entry $v_{ij}$ of the data matrix, an $R_y(θ_{ij})$ gate is applied, where $j$ corresponds to the feature index and $i$ to the record index.

The rotation angle is defined as:

$θ_{ij}​=2\arcsin{\left(\dfrac{v_{ij}}{r_{\max}}\right)}$

where
$r_{\max}$​ is the maximum absolute value in the matrix

$r_{\max}​=\max_{i,\ j}∣v_{ij}∣$

# Circuit

![alt text](<circuit.png>)

# Inner Product

## State


$|\psi_0 \rangle = | 0^{\otimes c}0^{\otimes d}00 \rangle$

$|\psi_1 \rangle = H^{\otimes c} \otimes H^{\otimes d}\otimes I ^{\otimes 2} |\psi_0\rangle = |ji00\rangle $

$|\psi_2\rangle = I^{\otimes 3} \otimes H |\psi_1\rangle = \dfrac{1}{\sqrt{2}}\left( |ji00\rangle + |ji01\rangle \right)$

$|\psi_3\rangle =  C_{A}^{0}(O_\phi)C_{A}^{1}(O_\theta) |\psi_2\rangle = \dfrac{1}{\sqrt{2}}\left( |ji\phi0\rangle + |ji\theta1\rangle \right)$

$|\psi_4\rangle = I^{\otimes 3} \otimes H |\psi_3\rangle = \dfrac{1}{2} \left( |ji\phi0\rangle + |ji\phi1\rangle + |ji\theta0\rangle - |ji\theta1\rangle \right)$

$\quad = \dfrac{1}{2} \left( |ji\phi0\rangle + |ji\theta0\rangle + |ji\phi1\rangle - |ji\theta1\rangle \right)$

$\quad = \dfrac{1}{2} \left[ (|ji\phi\rangle + |ji\theta\rangle)|0\rangle + (|ji\phi\rangle - |ji\theta\rangle)|1\rangle \right]$

## Probability


$Pr(0) = \dfrac{1}{4} (\langle ji\phi| + \langle ji\theta|)(|ji\phi\rangle + |ji\theta\rangle)$

$\quad = \dfrac{1}{4}(\langle ji\phi|ji\phi\rangle + \langle ji\phi|ji\theta\rangle + \langle ji\theta|ji\phi\rangle +  \langle ji\theta|ji\theta\rangle)$

$\quad = \dfrac{1}{4}(\langle ji|ji\rangle \langle\phi|\phi\rangle + \langle ji|ji\rangle \langle\phi|\theta\rangle + \langle ji|ji\rangle \langle\theta|\phi\rangle + \langle ji|ji\rangle \langle\theta|\theta\rangle)$

$\quad = \dfrac{1}{4}(\langle\phi|\phi\rangle + \langle\phi|\theta\rangle + \langle\theta|\phi\rangle + \langle\theta|\theta\rangle)$

$\quad = \dfrac{1}{4}(2+ \langle\phi|\theta\rangle + \langle\theta|\phi\rangle)$

$\quad = \dfrac{1}{4}(2 + 2\langle\phi|\theta\rangle)$

$\quad = \dfrac{1}{2} + \dfrac{1}{2}\langle\phi|\theta\rangle$

$\therefore \langle\phi|\theta\rangle = 2Pr(0) - 1$

## Distance

$d_e^2 = \|\theta\|^2 + \|\phi\|^2 - 2\langle\theta|\phi\rangle$

$\quad = 2 - 2(Pr(0) - 1)$

$\quad = 2 - 4Pr(0) + 2$

$\quad = 4 - 4Pr(0)$

$\quad = 4(1 - Pr(0))$

# Hybrid Euclidean Clasifier

## Imports

In [4]:
import pennylane as qml
from pennylane import numpy as np
from numpy.typing import NDArray

## Utils

In [5]:

def logT(num: int) -> tuple[int, int]:
    return num, int(np.ceil(np.log2(num))) if num > 1 else 1

def ctrlGen(d: int) -> list[str]:
    P = []

    logd: int = logT(d)[1]
    numsD = set([i for i in range(logd)])
    for i in range(d):
        B = [int(p) for p in bin(i)[2:]]
        append = int(logd - len(B))
        for _ in range(append): 
            B.insert(0, 0)

        p = ''.join(str(_) for _ in B)
        P.append(p)

    return P

def probs(counts: dict, nStates: int) -> NDArray[np.floating]:
    probs_arr = np.array([], dtype=np.float64)
    
    counts_idx = {}
    counts_match = {}
    
    for state, count in counts.items():
        if isinstance(state, tuple):
            bit_string = "".join(str(s) for s in state)
        elif isinstance(state, str):
            bit_string = state
        else:
            bit_string = bin(state)[2:]

        if not bit_string: continue

        idx_bits = bit_string[:-1]
        ancilla = bit_string[-1]
        
        idx_decimal = int(idx_bits, 2) if idx_bits else 0
        
        counts_idx[idx_decimal] = counts_idx.get(idx_decimal, 0) + count
        if ancilla == '0':
            counts_match[idx_decimal] = counts_match.get(idx_decimal, 0) + count

    for i in range(nStates):
        total = counts_idx.get(i, 0)
        match = counts_match.get(i, 0)
        
        prob = match / total if total > 0 else 0.0
        probs_arr = np.append(probs_arr, prob)
        
    return probs_arr

def dist(
        probs_arr: NDArray[np.floating], 
        method: str = 'hadamard', 
        norms2: NDArray[np.floating] = None
    ) -> NDArray[np.floating]:
    method = method.lower()
    if method not in ['hadamard', 'swap']:
        raise ValueError(f'Method {method} not recognized.')
    
    dists = np.array([], dtype=np.float64)
    if norms2 is None or len(norms2) == 0:
        return dists
        
    vec = norms2[0]
    vecs = norms2[1:]
    for i in range(len(probs_arr)):
        z = 1 if method == 'hadamard' else vec + vecs[i]
        dist_val = 4 * z * (1 - probs_arr[i])
        dists = np.append(dists, dist_val)

    return dists

## Inner product

In [ ]:
class InnerProduct:
    def __init__(self, X: NDArray[np.floating], y: NDArray[np.floating]) -> None:
        self.X = X
        self.labels = y
        self.Y = np.unique(self.labels)
        self.C, self.norms2 = self.centroids()
        
        self.log_k = int(np.ceil(np.log2(self.C.shape[0]))) if self.C.shape[0] > 1 else 1
        self.n, self.log_n = logT(self.X.shape[0])
        self.d, self.log_d = logT(self.X.shape[1])
        
        self.P = ctrlGen(self.d)
        self.Q = ctrlGen(self.C.shape[0])
        
        self.dev = qml.device(
            "default.qubit", 
            wires=self.log_k + self.log_d + 2
        )
        self.qnode = qml.QNode(self.run, self.dev)
        
        self.reg_J = [i for i in range(self.log_k)]
        self.reg_I = [i for i in range(self.log_k, self.log_k + self.log_d)]
        self.reg_V = self.log_k + self.log_d
        self.reg_A = self.log_k + self.log_d + 1

    def run(self, vec: NDArray[np.floating]):
        for i in self.reg_J:
            qml.Hadamard(wires=i)
        for i in self.reg_I:
            qml.Hadamard(wires=i)
        qml.Hadamard(wires=self.reg_A)
        
        max_X = np.max(np.abs(self.X))
        if max_X == 0:
            max_X = 1.0

        for i, p in enumerate(self.P):
            # todo: Codify the input vector
            # 1. Verify the index i is lower than the lenght of vec
            
            # 2. Normalize the value of the cell
            # 3. Make sure the value is in [-1, 1] (you may use np.clip).
            # 4. Calculate theta
            # 5. Define the values for the control 
            #    Nota: Conbert str into int.
            #          Keep in mind that p works for QRAM and you need an open ctrl
            # 6. Apply the controlled RY gates
            #    -> Ctrl wires are: self.reg_I[:] + [self.reg_A]
            #    -> Ctrl values were calculated earlier
            pass
            
        for j, q in enumerate(self.Q):
            for i, p in enumerate(self.P):
                # todo: Codify the centroids matrix
                # 1. Verify the index i is lower than the lenght of centroids length
                # 2. Normalize the value of the cell
                # 3. Make sure the value is in [-1, 1] (you may use np.clip).
                # 4. Calculate theta
                # 5. Define the values for the control 
                #    Nota: Conbert str into int.
                #          Keep in mind that p and q work for QRAM and you need a closed ctrl
                # 6. Apply the controlled RY gates}
                #    -> Ctrl wires are: self.reg_J[:] + self.reg_I[:] + [self.reg_A]
                #    -> Ctrl values were calculated earlier
                pass

        qml.Hadamard(wires=self.reg_A)
        return qml.counts(wires=self.reg_J[::]+[self.reg_A])

    def centroids(self) -> tuple[NDArray[np.float64], NDArray[np.float64]]: 
        centroids = np.array([ self.X[self.labels==c].mean(axis=0) for c in self.Y], dtype=np.float64)
        normas = np.array([np.linalg.norm(c) ** 2 for c in centroids], dtype=np.float64)
        return centroids, normas

## Quantum Euclidean Classifier

In [7]:
class QEuclidean:
    def __init__(self, metric="hadamard"):
        self.metric = metric

    def fit(self, X: NDArray[np.floating], y: NDArray[np.floating]) -> None:
        self.X = X
        self.y = y
        self.circuit = InnerProduct(self.X, self.y)

    def predict(self, vec: NDArray[np.floating]) -> float:
        counts = self.circuit.qnode(vec, shots=1024)
        n_classes = self.circuit.C.shape[0]
        
        probs_arr = probs(counts, n_classes)
        
        dists = []
        for i in range(n_classes):
            Z = self.circuit.norms2[i] + np.linalg.norm(vec) ** 2
            prob_val = probs_arr[i] if i < len(probs_arr) else 0.0
            
            dist_val = 4 * Z * (1 - prob_val) 
            dists.append(dist_val)

        dists = np.array(dists)
        
        best_idx = np.argmin(dists)
        return self.circuit.Y[best_idx]

## Running

In [8]:
print("--- Inicializando datos para el clasificador Híbrido ---")

X_train = np.array([
    [0.2, 0.3],
    [0.3, 0.2],
    [0.1, 0.4],
    [0.8, 0.9],
    [0.9, 0.8],
    [0.9, 0.9]
])
y_train = np.array([0, 0, 0, 1, 1, 1])

X_test = np.array([
    [0.25, 0.25],
    [0.85, 0.85]
])
y_test = np.array([0, 1])

print("\nTraining data:")
print(X_train)
print("Traning labels:", y_train)

print("\n--- Model training ---")
clf = QEuclidean()
clf.fit(X_train, y_train)
print("Training completed.")
print("Centroids:")
print(clf.circuit.C)

print("\n--- Model evaluation ---")
for i, x in enumerate(X_test):
    pred = clf.predict(x)
    print(f"Sample {i+1} {x}: Real = {y_test[i]} | Prediction = {pred}")
    if pred == y_test[i]:
        print("Right")
    else:
        print("wrong")

--- Inicializando datos para el clasificador Híbrido ---

Training data:
[[0.2 0.3]
 [0.3 0.2]
 [0.1 0.4]
 [0.8 0.9]
 [0.9 0.8]
 [0.9 0.9]]
Traning labels: [0 0 0 1 1 1]

--- Model training ---
Training completed.
Centroids:
[[0.2        0.3       ]
 [0.86666667 0.86666667]]

--- Model evaluation ---
Sample 1 [0.25 0.25]: Real = 0 | Prediction = 0
Right
Sample 2 [0.85 0.85]: Real = 1 | Prediction = 0
wrong


/Users/leopoldomorante/QuiskitLeo/anaconda3/envs/quantum_env/lib/python3.12/site-packages/pennylane/workflow/qnode.py:808: PennyLaneDeprecationWarning: Specifying 'shots' when executing a QNode is deprecated and will be removed in v0.44. Please set shots on QNode initialization, or use qml.set_shots instead.
  shots = self._get_shots(kwargs)
